# SQL Project — Book Service Analysis

During the pandemic, reading habits grew and new startups emerged to serve readers. We received the database of a competing service — books, authors, publishers, ratings and reviews — to extract insights that support the proposal for a new product.

This notebook answers five business questions in SQL, using pandas only to run queries and display results.

> **Note on column names.** SQL aliases are kept in Portuguese, as in the original run:
>
> | Alias | Meaning |
> |---|---|
> | `qtd_livros` | number of books |
> | `qtd_avaliacoes` | number of reviews |
> | `qtd_classificacoes` | number of ratings |
> | `classificacao_media` | average rating |
> | `media_livro` / `media_autor` | average rating per book / per author |
> | `media_avaliacoes` | average number of reviews |

## Step 1 — Study objectives

Understand the catalog and user behaviour to inform product decisions:

1. Size the modern catalog (books published after 2000-01-01).
2. Measure engagement per book (number of reviews and average rating).
3. Identify the most relevant publisher among books over 50 pages.
4. Find the best-rated authors (critical mass of ≥ 50 ratings).
5. Measure the engagement of the most active users (> 50 books rated).

In [1]:
# import libraries
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# database access: credentials live in a local .env file (see .env.example),
# never in the notebook
load_dotenv()

connection_url = URL.create(
    'postgresql',
    username=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'],
    host=os.environ['DB_HOST'],
    port=int(os.environ.get('DB_PORT', 5432)),
    database=os.environ['DB_NAME'],
)
engine = create_engine(connection_url, connect_args={'sslmode': 'require'})

In [2]:
# helper used by every query in the notebook
def run_query(query):
    '''Run a SQL query and return the result as a DataFrame.'''
    with engine.connect() as conn:
        return pd.read_sql(text(query), con=conn)

## Step 2 — Connection and table exploration

A sample of the five tables to confirm types, keys and grain before querying.

In [3]:
for table in ['books', 'authors', 'publishers', 'ratings', 'reviews']:
    display(run_query(f'SELECT * FROM {table} LIMIT 5'))

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


## Step 3 — Books released after 2000-01-01

**Task:** count how many books were published after 1 January 2000.
**Approach:** `COUNT` over `books`, filtering `publication_date` with `WHERE`.

In [4]:
query = '''
    SELECT COUNT(book_id) AS qtd_livros
    FROM books WHERE publication_date > '2000-01-01'
    '''
run_query(query)

,qtd_livros
0,819


**Result and conclusion:** 819 books. The catalog is heavily concentrated in recent publications (819 of 1,000 books, i.e. 82%) — it serves readers of new releases well, not readers of classics. I read "after" as `>` (exclusive); see the Decision Log.

## Step 4 — Reviews and average rating per book

**Task:** for each book, the number of reviews and the average rating.
**Approach:** `books` as the base, `LEFT JOIN` to `reviews` and `ratings`. `COUNT(DISTINCT review_id)` stops the fan-out from the two joins inflating the count; `AVG(rating)` is unaffected because the duplication is uniform within each book.

In [5]:
query = '''
    SELECT
        b.book_id,
        b.title,
        COUNT(DISTINCT rev.review_id) AS qtd_avaliacoes,
        AVG(rat.rating) AS classificacao_media
    FROM books AS b
    LEFT JOIN reviews AS rev ON b.book_id = rev.book_id
    LEFT JOIN ratings AS rat ON b.book_id = rat.book_id
    GROUP BY b.book_id, b.title
    '''
run_query(query)

,book_id,title,qtd_avaliacoes,classificacao_media
0,1,'Salem's Lot,2,3.666667
1,2,1 000 Places to See Before You Die,1,2.500000
2,3,13 Little Blue Envelopes (Little Blue Envelope...,3,4.666667
3,4,1491: New Revelations of the Americas Before C...,2,4.500000
4,5,1776,4,4.000000
...,...,...,...,...
995,996,Wyrd Sisters (Discworld #6; Witches #2),3,3.666667
996,997,Xenocide (Ender's Saga #3),3,3.400000
997,998,Year of Wonders,4,3.200000
998,999,You Suck (A Love Story #2),2,4.500000


In [6]:
resultado = run_query(query)
print('NULL por coluna:')
print(resultado.isna().sum())
print('\nLivros com zero avaliações:', (resultado['qtd_avaliacoes'] == 0).sum())

NULL por coluna:
book_id                0
title                  0
qtd_avaliacoes         0
classificacao_media    0
dtype: int64

Livros com zero avaliações: 6


**Result and conclusion:** 1,000 books (the full catalog). 6 have no written review — but all 1,000 have a rating (zero NULLs in `classificacao_media`). In other words, rating and reviewing are independent forms of engagement, and written reviews are the scarcer channel. Keeping those 6 books in the result was only possible with the `LEFT JOIN`.

## Step 5 — Publisher with the most books (> 50 pages)

**Task:** identify the publisher that released the largest number of books with more than 50 pages, excluding brochures and similar short publications.
**Approach:** `JOIN` between `books` and `publishers` on `publisher_id`, filtering `num_pages > 50` with `WHERE`. I group by publisher, sort by count in descending order and keep the top three with `LIMIT 3` — the leader answers the task; the next two show the gap.

In [7]:
query = '''
    SELECT
        p.publisher,
        COUNT(b.book_id) AS qtd_livros
    FROM books AS b
    JOIN publishers AS p ON b.publisher_id = p.publisher_id
    WHERE b.num_pages > 50
    GROUP BY p.publisher_id, p.publisher
    ORDER BY qtd_livros DESC
    LIMIT 3
'''
run_query(query)

,publisher,qtd_livros
0,Penguin Books,42
1,Vintage,31
2,Grand Central Publishing,25


**Result and conclusion:** Penguin Books leads with 42 books over 50 pages, ahead of Vintage (31) and Grand Central Publishing (25). The result is consistent with the publisher's size in the market — a large, generalist house at the top is a sanity check on the query. For the product, it signals that partnerships or curation with major publishers reach the largest share of the relevant catalog.

## Step 6 — Author with the highest average rating (≥ 50 ratings)

**Task:** identify the author with the highest average rating, considering only books with at least 50 ratings.
**Approach:** a two-layer subquery. The inner layer computes the average and the count of ratings **per book**, filtering with `HAVING COUNT >= 50`. The outer layer treats that result as a table, joins it to `authors` and takes the average of the averages **per author**.

Before aggregating by author, I isolate the books with at least 50 ratings — the base of the subquery. These are the books that clear the floor and feed the per-author calculation below.

In [8]:
# Books that cleared the 50+ ratings floor (base of the subquery below)
query = '''
    SELECT
        b.book_id,
        b.author_id,
        AVG(rat.rating) AS media_livro,
        COUNT(rat.rating_id) AS qtd_classificacoes
    FROM books AS b
    JOIN ratings AS rat ON b.book_id = rat.book_id
    GROUP BY b.book_id, b.author_id
    HAVING COUNT(rat.rating_id) >= 50
    ORDER BY qtd_classificacoes DESC
'''
run_query(query)

,book_id,author_id,media_livro,qtd_classificacoes
0,948,554,3.662500,160
1,750,240,4.125000,88
2,673,235,3.825581,86
3,75,106,3.678571,84
4,302,236,4.414634,82
5,299,236,4.287500,80
6,301,236,4.186667,75
7,79,195,3.729730,74
8,722,240,4.391892,74
9,300,236,4.246575,73


In [9]:
query = '''
    SELECT
        au.author,
        AVG(sub.media_livro) AS media_autor
    FROM (
        SELECT
            b.author_id,
            b.book_id,
            AVG(rat.rating) AS media_livro,
            COUNT(rat.rating_id) AS qtd_classificacoes
        FROM books AS b
        JOIN ratings AS rat ON b.book_id = rat.book_id
        GROUP BY b.book_id, b.author_id
        HAVING COUNT(rat.rating_id) >= 50
    ) AS sub
    JOIN authors AS au ON au.author_id = sub.author_id
    GROUP BY au.author_id, au.author
    ORDER BY media_autor DESC
    LIMIT 3
'''
run_query(query)

,author,media_autor
0,J.K. Rowling/Mary GrandPré,4.283844
1,Markus Zusak/Cao Xuân Việt Khương,4.264151
2,J.R.R. Tolkien,4.258446


**Result and conclusion:** J.K. Rowling/Mary GrandPré leads with an average of ~4.28. Only 19 of the 1,000 books cleared the 50-rating floor — the catalog has a long tail, and the floor protects the ranking from books whose average rests on a handful of ratings. The author field concatenates collaborators (author/illustrator), a feature of the source table.

## Step 7 — Average number of reviews among very active users (> 50 books)

**Task:** find the average number of reviews among users who rated more than 50 books.
**Approach:** three layers. The innermost isolates users who rated more than 50 books (`HAVING COUNT(DISTINCT book_id) > 50`). The middle layer counts how many reviews each of those users wrote. The outer layer averages those counts.

In [10]:
query = '''
    SELECT AVG(sub.qtd_avaliacoes) AS media_avaliacoes
    FROM (
        SELECT
            rev.username,
            COUNT(rev.review_id) AS qtd_avaliacoes
        FROM reviews AS rev
        WHERE rev.username IN (
            SELECT rat.username
            FROM ratings AS rat
            GROUP BY rat.username
            HAVING COUNT(DISTINCT rat.book_id) > 50
        )
        GROUP BY rev.username
    ) AS sub
'''
run_query(query)

,media_avaliacoes
0,24.333333


**Result and conclusion:** only 6 users rated more than 50 books — the power users of the base. All six also write reviews, averaging ~24.3 reviews each. This group is hyper-engaged in both channels (ratings and text), the most valuable profile for the product: it generates structured data and content at the same time.

## Analytical Decision Log

A record of the methodological choices that required my judgement. Entries tagged **[autoral]** are my own calls; steps without an entry were a direct execution of the brief.

**Step 3 — [autoral]** I read "after 1 January 2000" as `>` (exclusive), so books dated 2000-01-01 are left out. Result: 819.

**Step 4 — [autoral]** I chose `LEFT JOIN` over `INNER`. The check confirmed 6 books without a written review — with an `INNER JOIN` on reviews they would drop out and the result would have 994 rows, not 1,000. The `LEFT` join was necessary to keep the full catalog.

**Step 6 — [autoral]** I computed the "mean of per-book means" (each book weighted equally), the literal reading of the brief. The alternative — a mean weighted by number of ratings — would give a different result, but I kept to the literal interpretation of the task.

**Step 7 — [autoral]** I used `COUNT(DISTINCT rat.book_id)` instead of `COUNT(rating_id)` to count books, not ratings. If the system lets a user rate the same book more than once (e.g. a re-rating stored as a new row), the distinct keeps the count faithful to the brief ("more than 50 books"). I checked that all 6 users have at least one review, so none of them was left out of the average for being absent from the text channel.

## Step 8 — Overall conclusions

The catalog is modern and broad: 819 post-2000 books in a collection of 1,000. Engagement, however, is deeply uneven — most books have few reviews, and only 19 reached a critical mass of 50+ ratings. Among authors with that mass, J.K. Rowling/Mary GrandPré leads (~4.28). On the supply side, Penguin Books dominates the relevant catalog (> 50 pages) with 42 titles.

The most actionable finding is about users: only 6 people rated more than 50 books, and this same group writes ~24 reviews on average. The base is sustained by a small, hyper-engaged elite. For the new product, this suggests two fronts: (1) cultivate and retain these power users, who generate most of the data signal; (2) reduce friction for the median user to rate, widening engagement beyond the current core.